# 04 — Financial trends

**Question:** Are company fundamentals improving, deteriorating or stable over 2016–2025? Emphasis is on the latest three to five years. This is descriptive research, not a prediction.


## 1. Inputs and timing

Use the approved cleaned annual figures available by 31 December 2025. Historical charts use hindsight; a historical trading signal would need each row's `available_date`.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
project_root = Path.cwd().resolve()
if project_root.name == "notebooks": project_root = project_root.parent
if str(project_root) not in sys.path: sys.path.insert(0, str(project_root))
from src.data_loader import load_cleaned_data
from src.research.fundamentals import calculate_fundamentals, latest_fundamentals, compare_periods
annual, _ = load_cleaned_data(project_root / "data" / "Cleaned")
annual = annual.loc[annual.available_date.le(pd.Timestamp("2025-12-31"))].copy()
fundamentals = calculate_fundamentals(annual)
latest = latest_fundamentals(annual)
assert len(latest) == 20


## 2. Multi-year comparison

Amounts are shown directly so sign changes remain visible. Compare revenue with profit margin, FCF, net debt and shares outstanding.


In [ ]:
periods = compare_periods(annual, 2020, 2025)
columns = ["ticker", "sales_gbp_m_2020", "sales_gbp_m_2025", "profit_gbp_m_2020", "profit_gbp_m_2025", "profit_margin_2020", "profit_margin_2025", "free_cash_flow_gbp_m_2020", "free_cash_flow_gbp_m_2025", "net_debt_gbp_m_2020", "net_debt_gbp_m_2025", "shares_m_2020", "shares_m_2025"]
display(periods[columns].set_index("ticker").round(3))


## 3. Revenue trends

Each company is indexed to its own 2016 sales value. The sector panels support comparison without obscuring individual lines.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharex=True)
for ax, (sector, subset) in zip(axes.flat, fundamentals.groupby("sector")):
    for ticker, series in subset.groupby("ticker"):
        ax.plot(series.year, series.sales_gbp_m / series.sales_gbp_m.iloc[0] * 100, label=ticker)
    ax.set(title=sector, ylabel="Sales index (2016 = 100)", xlabel="Fiscal year")
    ax.legend(fontsize=8, ncol=2)
axes.flat[-1].axis("off")
fig.suptitle("Revenue history by sector")
fig.tight_layout()
plt.show()


## 4. Recent profit, cash and debt changes

This 2022–2025 comparison highlights revenue growth accompanied by deteriorating margins or FCF. Profit increasing while FCF falls is a review flag; it does not identify the cause.


In [ ]:
recent = compare_periods(annual, 2022, 2025)
recent["sales_change_gbp_m"] = recent.sales_gbp_m_2025 - recent.sales_gbp_m_2022
recent["margin_change_pp"] = (recent.profit_margin_2025 - recent.profit_margin_2022) * 100
recent["profit_change_gbp_m"] = recent.profit_gbp_m_2025 - recent.profit_gbp_m_2022
recent["fcf_change_gbp_m"] = recent.free_cash_flow_gbp_m_2025 - recent.free_cash_flow_gbp_m_2022
recent["net_debt_change_gbp_m"] = recent.net_debt_gbp_m_2025 - recent.net_debt_gbp_m_2022
recent["share_count_change_pct"] = (recent.shares_m_2025 / recent.shares_m_2022 - 1) * 100
recent["profit_up_fcf_down"] = recent.profit_change_gbp_m.gt(0) & recent.fcf_change_gbp_m.lt(0)
display(recent[["ticker", "sales_change_gbp_m", "margin_change_pp", "profit_change_gbp_m", "fcf_change_gbp_m", "net_debt_change_gbp_m", "share_count_change_pct", "profit_up_fcf_down"]].set_index("ticker").round(2))


In [ ]:
focus = fundamentals.loc[fundamentals.year.ge(2021)]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ticker in ["QCK", "BSL", "CDB", "CRM"]:
    series = focus.loc[focus.ticker.eq(ticker)]
    axes[0].plot(series.year, series.profit_margin * 100, marker="o", label=ticker)
    axes[1].plot(series.year, series.free_cash_flow_gbp_m, marker="o", label=ticker)
axes[0].set(title="Selected profit margins", ylabel="Profit margin (%)", xlabel="Fiscal year")
axes[1].set(title="Selected cash flow", ylabel="FCF (GBP m)", xlabel="Fiscal year")
for ax in axes: ax.grid(alpha=.25); ax.legend()
fig.tight_layout(); plt.show()


## 5. Lender and insurer trends

Credit indicators apply only to CDB and BSL. CRM uses the insurer fields. These measures cannot be placed on a single industrial leverage ranking.


In [ ]:
special = fundamentals.loc[fundamentals.ticker.isin(["CDB", "BSL", "CRM"]) & fundamentals.year.ge(2021), ["ticker", "year", "loans_overdue", "capital_cushion", "claims_ratio", "capital_coverage"]]
display(special.set_index(["ticker", "year"]).round(3))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ticker in ["CDB", "BSL"]:
    series = special.loc[special.ticker.eq(ticker)]
    axes[0].plot(series.year, series.loans_overdue * 100, label=f"{ticker} overdue")
    axes[0].plot(series.year, series.capital_cushion * 100, linestyle="--", label=f"{ticker} cushion")
crm = special.loc[special.ticker.eq("CRM")]
axes[1].plot(crm.year, crm.claims_ratio * 100, label="Claims ratio (%)")
axes[1].plot(crm.year, crm.capital_coverage * 100, label="Capital cover (×100)")
axes[0].set(title="Lender indicators", ylabel="Percent (%)", xlabel="Fiscal year")
axes[1].set(title="CRM indicators", ylabel="Percent or ratio ×100", xlabel="Fiscal year")
for ax in axes: ax.grid(alpha=.25); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()


## 6. Supported observations and questions

The table isolates rising sales with falling margin and rising profit with falling FCF. Examine working capital, investment, debt funding and share issuance before interpreting the flags. The 2025 data are provisional and trends are not guarantees.


In [ ]:
review = recent.loc[(recent.sales_change_gbp_m.gt(0) & recent.margin_change_pp.lt(0)) | recent.profit_up_fcf_down, ["ticker", "sales_change_gbp_m", "margin_change_pp", "profit_change_gbp_m", "fcf_change_gbp_m", "profit_up_fcf_down"]]
display(review.set_index("ticker").round(2))


In [ ]:
print("Tickers for further investigation:", ", ".join(review.ticker) if not review.empty else "none under these flags")
print("2022–2025: rising sales with lower margin:", ", ".join(recent.loc[recent.sales_change_gbp_m.gt(0) & recent.margin_change_pp.lt(0), "ticker"]))
print("2022–2025: profit rose while FCF fell:", ", ".join(recent.loc[recent.profit_up_fcf_down, "ticker"]))
